# Stage-A Stability Public Module (Dev + Lockbox)

This notebook is a lightweight public module for the Stage-A stability classifier.

It uses:
- the full public **679-sample** stability table with the final **30 selected features**
- a fixed **Dev/Lockbox split file**
- the frozen tuned model **`LGBM_TUNED_ROC.joblib`**
- SHAP analysis based on the frozen tuned model

The workflow intentionally excludes:
- raw data cleaning and label engineering
- the full hyperparameter search process
- active learning retraining rounds
- external validation notebooks

The public workflow includes:
1. load the full 679-sample table and a fixed Dev/Lockbox split
2. run simplified coarse modeling on **Dev only**
3. load the frozen tuned model and report **OOF performance on Dev**
4. run SHAP beeswarm and dependence plots using the frozen tuned model
5. evaluate the frozen tuned model on **Lockbox only**

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path.cwd()

DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "models"
OUT_DIR = ROOT / "outputs" / "stability_public_module"
FIG_DIR = OUT_DIR / "figures"
TAB_DIR = OUT_DIR / "tables"
REP_DIR = OUT_DIR / "reports"

for p in [FIG_DIR, TAB_DIR, REP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATA_CSV = DATA_DIR / "stability_stageA_679_best30.csv"
SPLIT_CSV = DATA_DIR / "stability_stageA_split.csv"
FEATURE_JSON = MODEL_DIR / "stability_features.json"
MODEL_JOBLIB = MODEL_DIR / "LGBM_TUNED_ROC.joblib"
META_JSON = MODEL_DIR / "stability_tuned_meta.json"

ID_COL = "MOFID"
TARGET_COL = "Label_Stab"
SUBSET_COL = "subset"
GROUP_COL = "group_key"     # optional
WEIGHT_COL = "sample_weight" # optional

SEED = 42
np.random.seed(SEED)

print("Data file:", DATA_CSV)
print("Split file:", SPLIT_CSV)
print("Feature file:", FEATURE_JSON)
print("Model file:", MODEL_JOBLIB)
print("Meta file:", META_JSON)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.unicode_minus": False,
    "figure.dpi": 140,
    "savefig.dpi": 450,
    "savefig.bbox": "tight",
})

def save_df(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print("Saved:", path)

def save_json(obj: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved:", path)

def save_fig(fig, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path)
    print("Saved:", path)
    plt.show()

def p1(y, est):
    if hasattr(est, "predict_proba"):
        return est.predict_proba(y)[:, 1]
    if hasattr(est, "decision_function"):
        z = est.decision_function(y)
        z = z[:, 1] if getattr(z, "ndim", 1) > 1 else z
        return 1.0 / (1.0 + np.exp(-z))
    return est.predict(y).astype(float)

In [ ]:
assert DATA_CSV.exists(), f"Missing: {DATA_CSV}"
assert SPLIT_CSV.exists(), f"Missing: {SPLIT_CSV}"
assert FEATURE_JSON.exists(), f"Missing: {FEATURE_JSON}"

df = pd.read_csv(DATA_CSV)
split_df = pd.read_csv(SPLIT_CSV)

with open(FEATURE_JSON, "r", encoding="utf-8") as f:
    final_feats = json.load(f)

assert ID_COL in df.columns, f"Missing {ID_COL} in data table"
assert TARGET_COL in df.columns, f"Missing {TARGET_COL} in data table"
assert ID_COL in split_df.columns, f"Missing {ID_COL} in split table"
assert SUBSET_COL in split_df.columns, f"Missing {SUBSET_COL} in split table"

missing_feats = [c for c in final_feats if c not in df.columns]
assert not missing_feats, f"Missing features in data table: {missing_feats[:10]}"

dat = df.merge(split_df, on=ID_COL, how="left", validate="one_to_one")
assert dat[SUBSET_COL].notna().all(), "Some samples are missing split assignment"

if WEIGHT_COL not in dat.columns:
    dat[WEIGHT_COL] = 1.0

if GROUP_COL not in dat.columns:
    if GROUP_COL in split_df.columns:
        dat[GROUP_COL] = dat[GROUP_COL]
    else:
        dat[GROUP_COL] = dat[ID_COL].astype(str)

dat[TARGET_COL] = dat[TARGET_COL].astype(int)
dat[WEIGHT_COL] = dat[WEIGHT_COL].astype(float)

dev_df = dat[dat[SUBSET_COL].str.lower() == "dev"].copy()
lock_df = dat[dat[SUBSET_COL].str.lower().isin(["lockbox", "lock"])].copy()

assert len(dev_df) > 0, "Dev subset is empty"
assert len(lock_df) > 0, "Lockbox subset is empty"

X_dev = dev_df[final_feats].copy()
y_dev = dev_df[TARGET_COL].to_numpy()
w_dev = dev_df[WEIGHT_COL].to_numpy()
g_dev = dev_df[GROUP_COL].astype(str).to_numpy()

X_lock = lock_df[final_feats].copy()
y_lock = lock_df[TARGET_COL].to_numpy()
w_lock = lock_df[WEIGHT_COL].to_numpy()
g_lock = lock_df[GROUP_COL].astype(str).to_numpy()

summary = pd.DataFrame({
    "subset": ["Full", "Dev", "Lockbox"],
    "n_samples": [len(dat), len(dev_df), len(lock_df)],
    "positive_rate": [dat[TARGET_COL].mean(), dev_df[TARGET_COL].mean(), lock_df[TARGET_COL].mean()],
    "n_groups": [dat[GROUP_COL].nunique(), dev_df[GROUP_COL].nunique(), lock_df[GROUP_COL].nunique()],
})
display(summary)
save_df(summary, TAB_DIR / "subset_summary.csv")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.2))

ratio_df = pd.DataFrame({
    "subset": ["Dev", "Lockbox"],
    "stable_rate": [dev_df[TARGET_COL].mean(), lock_df[TARGET_COL].mean()]
})
sns.barplot(data=ratio_df, x="subset", y="stable_rate", ax=axes[0], color="#7CCBA2")
axes[0].set_ylim(0, 1)
axes[0].set_title("Positive-class ratio")
axes[0].set_ylabel("Stable fraction")

grp_df = pd.DataFrame({
    "subset": ["Dev", "Lockbox"],
    "n_groups": [dev_df[GROUP_COL].nunique(), lock_df[GROUP_COL].nunique()]
})
sns.barplot(data=grp_df, x="subset", y="n_groups", ax=axes[1], color="#5DA5DA")
axes[1].set_title("Group coverage")
axes[1].set_ylabel("Number of unique groups")

save_fig(fig, FIG_DIR / "subset_overview.png")

## Part 1 — Simplified coarse modeling on Dev only

This step uses only the fixed Dev subset.  
The split is never redrawn in the public notebook.

In [ ]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier
from lightgbm import LGBMClassifier
import inspect

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

def supports_sample_weight(est):
    try:
        return "sample_weight" in inspect.signature(est.fit).parameters
    except Exception:
        return False

def make_splitter(y, groups=None, n_splits=5, seed=42):
    if groups is not None and pd.Series(groups).nunique() >= n_splits:
        return "group", GroupKFold(n_splits=n_splits)
    return "stratified", StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

def score_binary(y_true, p, thr=0.5):
    y_hat = (np.asarray(p) >= thr).astype(int)
    return {
        "ACC": accuracy_score(y_true, y_hat),
        "BACC": balanced_accuracy_score(y_true, y_hat),
        "F1": f1_score(y_true, y_hat),
        "ROC_AUC": roc_auc_score(y_true, p),
        "PR_AUC": average_precision_score(y_true, p),
    }

def fit_one(est, Xtr, ytr, wtr=None):
    if supports_sample_weight(est) and wtr is not None:
        return est.fit(Xtr, ytr, sample_weight=wtr)
    return est.fit(Xtr, ytr)

def prob_one(est, Xva):
    if hasattr(est, "predict_proba"):
        return est.predict_proba(Xva)[:, 1]
    if hasattr(est, "decision_function"):
        z = est.decision_function(Xva)
        z = z[:, 1] if getattr(z, "ndim", 1) > 1 else z
        return 1.0 / (1.0 + np.exp(-z))
    return est.predict(Xva).astype(float)

def eval_cv(est, X, y, groups=None, weights=None, n_splits=5, seed=42):
    mode, cv = make_splitter(y, groups=groups, n_splits=n_splits, seed=seed)
    rows = []
    idx = np.arange(len(y))
    split_iter = cv.split(idx, y, groups) if mode == "group" else cv.split(idx, y)
    for fold, (tr, va) in enumerate(split_iter, 1):
        model = clone(est)
        model = fit_one(model, X.iloc[tr], y[tr], None if weights is None else weights[tr])
        p = prob_one(model, X.iloc[va])
        row = score_binary(y[va], p, thr=0.5)
        row["fold"] = fold
        rows.append(row)
    fold_df = pd.DataFrame(rows)
    out = {f"{c}_mean": fold_df[c].mean() for c in ["ACC","BACC","F1","ROC_AUC","PR_AUC"]}
    out.update({f"{c}_std": fold_df[c].std(ddof=1) for c in ["ACC","BACC","F1","ROC_AUC","PR_AUC"]})
    return fold_df, out

baseline_models = {
    "Dummy": DummyClassifier(strategy="prior"),
    "Logistic": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=SEED))
    ]),
    "SVC": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(C=1.0, probability=True, class_weight="balanced", random_state=SEED))
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=7))
    ]),
    "DecisionTree": DecisionTreeClassifier(max_depth=6, random_state=SEED, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(
        n_estimators=400, random_state=SEED, n_jobs=-1, class_weight="balanced_subsample"
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=500, random_state=SEED, n_jobs=-1, class_weight="balanced_subsample"
    ),
    "AdaBoost": AdaBoostClassifier(n_estimators=250, learning_rate=0.05, random_state=SEED),
    "LightGBM": LGBMClassifier(
        random_state=SEED, n_estimators=300, learning_rate=0.05, num_leaves=31
    ),
}

if HAS_XGB:
    baseline_models["XGBoost"] = XGBClassifier(
        random_state=SEED,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        objective="binary:logistic",
        n_jobs=-1
    )

baseline_rows = []
baseline_fold_tables = {}
for name, est in baseline_models.items():
    fold_df, summary_row = eval_cv(est, X_dev, y_dev, groups=g_dev, weights=w_dev, n_splits=5, seed=SEED)
    summary_row["Model"] = name
    baseline_rows.append(summary_row)
    baseline_fold_tables[name] = fold_df

baseline_summary = pd.DataFrame(baseline_rows).sort_values("ROC_AUC_mean", ascending=False).reset_index(drop=True)
top3_models = baseline_summary["Model"].head(3).tolist()

display(baseline_summary)
print("Top-3 models:", top3_models)

save_df(baseline_summary, TAB_DIR / "dev_baseline_summary.csv")
for name, fdf in baseline_fold_tables.items():
    save_df(fdf.assign(Model=name), TAB_DIR / f"dev_baseline_folds_{name}.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.8))
plot_df = baseline_summary.copy()
sns.barplot(data=plot_df, x="ROC_AUC_mean", y="Model", color="#4C78A8", ax=ax)
ax.set_title("Simplified coarse modeling on Dev")
ax.set_xlabel("Mean ROC-AUC")
ax.set_ylabel("")
save_fig(fig, FIG_DIR / "dev_baseline_top_models_by_auc.png")

## Part 2 — Frozen tuned model and OOF performance on Dev

This step does not expose hyperparameter search.

The public module only uses:
- the saved frozen model `LGBM_TUNED_ROC.joblib`
- the saved metadata file `stability_tuned_meta.json`

OOF is computed on **Dev only**.

In [ ]:
import joblib
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.base import clone
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

class OOFCalibratedClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, base_estimator, n_splits=5, method="auto", random_state=42):
        self.base_estimator = base_estimator
        self.n_splits = n_splits
        self.method = method
        self.random_state = random_state
        self.calibrator_ = None
        self.base_fitted_ = None

    def fit(self, X, y, groups=None, sample_weight=None):
        X = np.asarray(X)
        y = np.asarray(y).astype(int)
        n = len(y)
        oof = np.zeros(n, dtype=float)

        gkf = GroupKFold(n_splits=self.n_splits)
        if groups is None:
            groups = np.arange(n)

        for tr, te in gkf.split(X, y, groups):
            est = clone(self.base_estimator)
            fit_kwargs = {}
            if sample_weight is not None and supports_sample_weight(est):
                fit_kwargs["sample_weight"] = sample_weight[tr]
            est.fit(X[tr], y[tr], **fit_kwargs)

            if hasattr(est, "predict_proba"):
                p = est.predict_proba(X[te])[:, 1]
            elif hasattr(est, "decision_function"):
                z = est.decision_function(X[te])
                p = 1.0 / (1.0 + np.exp(-z))
            else:
                p = est.predict(X[te]).astype(float)
            oof[te] = p

        method = "isotonic" if self.method == "auto" else self.method
        if method == "isotonic":
            try:
                cal = IsotonicRegression(out_of_bounds="clip")
                cal.fit(oof, y, sample_weight=sample_weight)
                self.calibrator_ = ("isotonic", cal)
            except Exception:
                logit = LogisticRegression(solver="lbfgs", max_iter=2000, class_weight="balanced")
                logit.fit(oof.reshape(-1, 1), y, sample_weight=sample_weight)
                self.calibrator_ = ("platt", logit)
        else:
            logit = LogisticRegression(solver="lbfgs", max_iter=2000, class_weight="balanced")
            logit.fit(oof.reshape(-1, 1), y, sample_weight=sample_weight)
            self.calibrator_ = ("platt", logit)

        est_full = clone(self.base_estimator)
        fit_kwargs = {}
        if sample_weight is not None and supports_sample_weight(est_full):
            fit_kwargs["sample_weight"] = sample_weight
        est_full.fit(X, y, **fit_kwargs)
        self.base_fitted_ = est_full
        return self

    def predict_proba(self, X):
        X = np.asarray(X)
        if hasattr(self.base_fitted_, "predict_proba"):
            p = self.base_fitted_.predict_proba(X)[:, 1]
        elif hasattr(self.base_fitted_, "decision_function"):
            z = self.base_fitted_.decision_function(X)
            p = 1.0 / (1.0 + np.exp(-z))
        else:
            p = self.base_fitted_.predict(X).astype(float)

        kind, cal = self.calibrator_
        if kind == "isotonic":
            pc = cal.predict(p)
        else:
            pc = cal.predict_proba(p.reshape(-1, 1))[:, 1]
        pc = np.clip(pc, 1e-9, 1 - 1e-9)
        return np.vstack([1 - pc, pc]).T

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

assert MODEL_JOBLIB.exists(), f"Missing: {MODEL_JOBLIB}"
assert META_JSON.exists(), f"Missing: {META_JSON}"

with open(META_JSON, "r", encoding="utf-8") as f:
    tuned_meta = json.load(f)

thr = float(tuned_meta["threshold"])
tuned_model = joblib.load(MODEL_JOBLIB)

print("Threshold:", thr)
print("Loaded object:", tuned_model.__class__.__name__)
print("Feature count:", len(final_feats))

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay

def oof_predict_public(est, X, y, groups=None, weights=None, n_splits=5, seed=42):
    mode, cv = make_splitter(y, groups=groups, n_splits=n_splits, seed=seed)
    idx = np.arange(len(y))
    oof_p = np.zeros(len(y), dtype=float)
    split_iter = cv.split(idx, y, groups) if mode == "group" else cv.split(idx, y)
    for fold, (tr, va) in enumerate(split_iter, 1):
        model = clone(est)
        if isinstance(model, OOFCalibratedClassifier):
            model.fit(X.iloc[tr], y[tr], groups=None if mode != "group" else groups[tr], sample_weight=None if weights is None else weights[tr])
            p = model.predict_proba(X.iloc[va])[:, 1]
        else:
            model = fit_one(model, X.iloc[tr], y[tr], None if weights is None else weights[tr])
            p = prob_one(model, X.iloc[va])
        oof_p[va] = p
    return oof_p

dev_oof_p = oof_predict_public(tuned_model, X_dev, y_dev, groups=g_dev, weights=w_dev, n_splits=5, seed=SEED)
dev_oof_yhat = (dev_oof_p >= thr).astype(int)

dev_oof_metrics = score_binary(y_dev, dev_oof_p, thr=thr)
dev_oof_metrics["threshold"] = thr
dev_oof_metrics["n_dev"] = int(len(y_dev))

display(pd.DataFrame([dev_oof_metrics]))
save_json(dev_oof_metrics, REP_DIR / "dev_oof_metrics.json")

oof_pred_df = pd.DataFrame({
    ID_COL: dev_df[ID_COL].values,
    "subset": "Dev",
    "y_true": y_dev,
    "p_oof": dev_oof_p,
    "y_hat": dev_oof_yhat
})
save_df(oof_pred_df, TAB_DIR / "dev_oof_predictions.csv")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15.6, 4.5))

fpr, tpr, _ = roc_curve(y_dev, dev_oof_p)
axes[0].plot(fpr, tpr, lw=2.2, color="#1f77b4")
axes[0].plot([0,1], [0,1], ls="--", lw=1.0, color="#999999")
axes[0].set_title(f"Dev OOF ROC (AUC={dev_oof_metrics['ROC_AUC']:.3f})")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")

prec, rec, _ = precision_recall_curve(y_dev, dev_oof_p)
axes[1].plot(rec, prec, lw=2.2, color="#2ca02c")
axes[1].set_title(f"Dev OOF PR (AP={dev_oof_metrics['PR_AUC']:.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")

cm = confusion_matrix(y_dev, dev_oof_yhat, labels=[0,1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(ax=axes[2], colorbar=False)
axes[2].set_title(f"Dev OOF confusion (thr={thr:.3f})")

save_fig(fig, FIG_DIR / "dev_oof_summary.png")

## Part 3 — SHAP analysis on the frozen tuned model

This step uses the frozen tuned model fitted on Dev only.

The SHAP plots are generated from the final frozen tuned model, which matches the SHAP source used in the manuscript.

In [ ]:
import shap

def unwrap_for_shap(obj):
    if hasattr(obj, "base_fitted_") and obj.base_fitted_ is not None:
        return obj.base_fitted_
    if hasattr(obj, "base_estimator") and obj.base_estimator is not None:
        return obj.base_estimator
    if hasattr(obj, "steps"):
        return obj.steps[-1][1]
    return obj

base_model_for_shap = unwrap_for_shap(tuned_model)
print("SHAP base model:", base_model_for_shap.__class__.__name__)

X_shap = X_dev.copy()

explainer = shap.TreeExplainer(
    model=base_model_for_shap,
    data=X_shap,
    feature_names=final_feats,
    model_output="probability",
    feature_perturbation="interventional",
)

shap_values_raw = explainer.shap_values(X_shap, check_additivity=False)

if isinstance(shap_values_raw, list):
    shap_values = np.asarray(shap_values_raw[1] if len(shap_values_raw) > 1 else shap_values_raw[0], dtype=float)
else:
    shap_values_raw = np.asarray(shap_values_raw)
    if shap_values_raw.ndim == 3 and shap_values_raw.shape[-1] >= 2:
        shap_values = np.asarray(shap_values_raw[..., 1], dtype=float)
    else:
        shap_values = np.asarray(shap_values_raw, dtype=float)

mean_abs = np.abs(shap_values).mean(axis=0)
shap_imp = pd.DataFrame({
    "feature": final_feats,
    "mean_abs_shap": mean_abs
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

display(shap_imp.head(15))
save_df(shap_imp, TAB_DIR / "shap_importance_dev.csv")
np.save(TAB_DIR / "shap_values_dev.npy", shap_values)

In [ ]:
fig = plt.figure(figsize=(8.6, 6.2))
shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=final_feats,
    max_display=min(15, len(final_feats)),
    show=False
)
plt.title("SHAP beeswarm — Stage-A stability")
save_fig(fig, FIG_DIR / "shap_beeswarm_dev.png")

In [ ]:
top_dep = shap_imp["feature"].head(min(6, len(shap_imp))).tolist()

for feat in top_dep:
    fig = plt.figure(figsize=(6.1, 4.8))
    shap.dependence_plot(
        feat,
        shap_values,
        X_shap,
        interaction_index=None,
        show=False
    )
    plt.title(f"SHAP dependence — {feat}")
    save_fig(fig, FIG_DIR / f"shap_dependence_{feat}.png")

## Part 4 — Lockbox evaluation

This step uses the same frozen tuned model and the fixed lockbox subset only.

In [ ]:
lock_p = tuned_model.predict_proba(X_lock)[:, 1]
lock_yhat = (lock_p >= thr).astype(int)

lock_metrics = score_binary(y_lock, lock_p, thr=thr)
lock_metrics["threshold"] = thr
lock_metrics["n_lockbox"] = int(len(y_lock))

display(pd.DataFrame([lock_metrics]))
save_json(lock_metrics, REP_DIR / "lockbox_metrics.json")

lock_pred_df = pd.DataFrame({
    ID_COL: lock_df[ID_COL].values,
    "subset": "Lockbox",
    "y_true": y_lock,
    "p_pred": lock_p,
    "y_hat": lock_yhat
})
save_df(lock_pred_df, TAB_DIR / "lockbox_predictions.csv")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15.6, 4.5))

fpr, tpr, _ = roc_curve(y_lock, lock_p)
axes[0].plot(fpr, tpr, lw=2.2, color="#1f77b4")
axes[0].plot([0,1], [0,1], ls="--", lw=1.0, color="#999999")
axes[0].set_title(f"Lockbox ROC (AUC={lock_metrics['ROC_AUC']:.3f})")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")

prec, rec, _ = precision_recall_curve(y_lock, lock_p)
axes[1].plot(rec, prec, lw=2.2, color="#2ca02c")
axes[1].set_title(f"Lockbox PR (AP={lock_metrics['PR_AUC']:.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")

cm = confusion_matrix(y_lock, lock_yhat, labels=[0,1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0,1])
disp.plot(ax=axes[2], colorbar=False)
axes[2].set_title(f"Lockbox confusion (thr={thr:.3f})")

save_fig(fig, FIG_DIR / "lockbox_summary.png")

In [ ]:
manifest = {
    "module": "Stage-A stability public module",
    "full_dataset_n": int(len(dat)),
    "dev_n": int(len(dev_df)),
    "lockbox_n": int(len(lock_df)),
    "feature_count": int(len(final_feats)),
    "frozen_model": str(MODEL_JOBLIB.name),
    "meta_file": str(META_JSON.name),
    "threshold": float(thr),
    "outputs": {
        "tables": [
            "subset_summary.csv",
            "dev_baseline_summary.csv",
            "dev_oof_predictions.csv",
            "shap_importance_dev.csv",
            "lockbox_predictions.csv"
        ],
        "reports": [
            "dev_oof_metrics.json",
            "lockbox_metrics.json"
        ]
    }
}
save_json(manifest, REP_DIR / "public_module_manifest.json")
display(pd.DataFrame({"output_folder":[str(OUT_DIR.resolve())]}))